# 梯度检查与调试

## 学习目标

用有限差分检查解析梯度，并建立定位 shape、非有限值和梯度规模问题的顺序。

## 概念模型

梯度检查逐个扰动参数，比较数值斜率与 backward 结果。它很慢，适合小输入和小网络，不用于训练。

## 逐步实现

按顺序运行下面的代码，并在每一步检查 shape、数值范围和中间结果。

In [ ]:
from pathlib import Path
import sys

course_dir = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd() / "07-deep-learning/fundamentals"
sys.path.insert(0, str(course_dir.resolve()))

import numpy as np
from from_scratch import CrossEntropyLoss, Linear, gradient_check

x = np.array([[0.2, -0.3], [0.7, 0.1]])
y = np.array([0, 1])
layer = Linear(2, 2, seed=5)
criterion = CrossEntropyLoss()

def current_loss():
    return criterion.forward(layer.forward(x), y)

current_loss()
layer.backward(criterion.backward())
relative_error = gradient_check(current_loss, layer.weight, layer.grad_weight.copy())
print("maximum relative error:", relative_error)
assert relative_error < 1e-6

In [ ]:
def inspect(name, value):
    print(name, "shape=", value.shape, "finite=", np.isfinite(value).all(), "norm=", np.linalg.norm(value))

inspect("weight", layer.weight)
inspect("grad_weight", layer.grad_weight)
# 调试顺序：数据/标签 → shape → loss → 梯度是否存在 → 梯度范围 → 参数是否更新。

## 检查点

相对误差接近 0 表示解析梯度与数值梯度一致。检查失败时先缩小到单层、少量样本和 float64。

## 试一试

故意把 `grad_weight` 乘以 2，观察梯度检查如何报告明显误差。

## 常见错误

在 ReLU 恰好为 0 的不可导点检查；epsilon 太大或太小；用完整数据集进行逐元素数值检查。